# Sensitivity Analyses

This script can be used to provide a robust measure of the robustness and stability of the AD Course Map model through Cross Validation 

In [6]:
import pandas as pd
import numpy as np
from leaspy import Leaspy, Data, AlgorithmSettings, __watermark__
from sklearn.model_selection import KFold

base_path = '../'
data_path = base_path + 'datasets/cognitive_scores.csv'
settings_path = base_path + 'utils/'

In [7]:
def crossvalidation(dataset, k):
    kf = KFold(n_splits=k, shuffle=True, random_state=0)
    id_list = np.array(dataset['ID'].unique())
    k_mae = dict()

    for index_train, index_val in kf.split(id_list):
        id_train = id_list[index_train]
        id_val = id_list[index_val]
        df_train = dataset[dataset['ID'].isin(id_train)]
        df_val = dataset[dataset['ID'].isin(id_val)]
        df_to_pred = df_val.groupby('ID').tail(1)
        df_pers = df_val[~df_val.index.isin(df_to_pred.index.tolist())]

        #from df to data
        data_train = Data.from_dataframe(df_train.set_index(['ID', 'TIME']))
        data_pers = Data.from_dataframe(df_pers.set_index(['ID', 'TIME']))
        df_to_pred = df_to_pred.set_index(['ID', 'TIME'])

        leaspy = Leaspy('logistic', 
                source_dimension=2,                  # number of dimensions for non-temporal inter-subject variability
                noise_model='gaussian_diagonal')     # estimate the residual noise scaling per feature

        # Algorithm Settings
        algo_settings = AlgorithmSettings.load(settings_path + 'algorithm_settings_calibration.json')
        # Fitting
        leaspy.fit(data_train, settings=algo_settings) 

        # Personalize step
        settings_personalization = AlgorithmSettings('scipy_minimize', seed=0)
        ip = leaspy.personalize(data_pers, settings_personalization)
        # Future predictions on biomarkers of patients who have had their biomarkers personalized
        predictions = leaspy.estimate(dict(zip(df_to_pred.index.get_level_values('ID'),df_to_pred.index.get_level_values('TIME'))), ip)

        # Create a dataframe from predictions
        prediction_temp = {k:v[0] for k, v in predictions.items()} # Make a well format dictionary to majke a dataframe
        df_predicted = pd.DataFrame.from_dict(prediction_temp, orient='index', columns=['MMSE', 'Memory', 'Language', 'Concentration', 'Praxis'])

        # Calculation of MAE for every biomarkers
        for x, var in enumerate(df_to_pred.columns):
            error = np.abs(np.asarray(df_to_pred[var]) - np.asarray(df_predicted[var])).mean()
            if var in k_mae.keys():
                k_mae[var].append(error)
            else:
                k_mae[var] = [error]

    return k_mae

In [8]:
# Read data
dataset = pd.read_csv(data_path)
dataset['ID'] = dataset['ID'].astype('str') # Make id as a string

#definition of a sub group
sub_n = '1000' # number of subjects
vis_n = dataset[dataset['ID']==sub_n].index[-1]+1 # number of visits (rows)
dataset = dataset.iloc[:vis_n]

In [ ]:
# Cross Validation
k_mae = crossvalidation(dataset, k=5)

In [ ]:
print('Mean kfold MAE')
for var in k_mae.keys():
    print(var+': '+ str(round(np.mean(k_mae[var]), 5)))